<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment: Building a Modular Data Sanitization & Exploration Engine**
# **E/23/355**

---
# Features

*  **Intelligent Data Loading** — handles null strings (?, N/A, NULL) and auto-converts columns.  
*   **Comprehensive Inspection** — dimensions, type breakdowns, statistical summaries.
* **Automated Cleaning** — mean/median/mode/constant imputation, duplicate removal, IQR outliers.
* **Advanced Scaling & Encoding** — Min-Max, Z-score, Robust; One-Hot, Ordinal, Uniform.
* **Interactive Visualizations** — Plotly violin, scatter, histogram, grouped bar charts.
* **Deep Statistical Insights**  — Pearson r, Cramér's V, Point-Biserial/Eta unified heatmap.

---

# Cell 1 — Package Installation








In [1]:
!pip install plotly scipy --quiet

---
# Cell 2 — Class Definitions

**Imports**

In [2]:
import io, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import chi2_contingency
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

---
# PlottingMethods Class

In [23]:
class PlottingMethods:
    """Granular chart generation — returns HTML-wrapped Plotly figures."""

    @staticmethod
    def _wrap(fig, title=""):
        if title: fig.update_layout(title_text=title)
        return {'status':'ok','fig':fig,
                'html':fig.to_html(full_html=False,include_plotlyjs='cdn')}

    @staticmethod
    def _err(msg):
        return {'status':'error','fig':None,'html':None,'message':msg}

    @staticmethod
    def display_image(result):
        """Render HTML figure in Colab."""
        if result.get('status')=='ok' and result.get('html'):
            display(HTML(result['html']))
        else:
            print(f"[PlottingMethods] {result.get('message','Unknown error')}")

    @staticmethod
    def get_methods_info():
        methods = [
            {'method':'plot_bar_chart', 'description':'Grouped/stacked bar chart'},
            {'method':'plot_pie_chart', 'description':'Responsive donut/pie chart'},
            {'method':'plot_histogram', 'description':'Histogram with custom bins'},
        ]
        return {'response': methods}

    def plot_bar_chart(self, x, y, data, color=None,
                       barmode='group', title=''):
        """Grouped or stacked bar chart."""
        if data is None or data.empty:
            return self._err("No data.")
        try:
            fig = px.bar(data, x=x, y=y, color=color,
                         barmode=barmode, template='plotly_white')
            return self._wrap(fig, title or f'{y} by {x}')
        except Exception as e:
            return self._err(str(e))

    def plot_pie_chart(self, names, values, data, hole=0.4, title=''):
        """Responsive donut / pie chart."""
        if data is None or data.empty:
            return self._err("No data.")
        try:
            fig = px.pie(data, names=names, values=values,
                         hole=hole, template='plotly_white')
            return self._wrap(fig, title or f'{values} split by {names}')
        except Exception as e:
            return self._err(str(e))

    def plot_histogram(self, x, data, bins=None, title=''):
        """Histogram with optional custom bin edges."""
        if data is None or data.empty:
            return self._err("No data.")
        try:
            fig = px.histogram(data, x=x, template='plotly_white')
            if bins:
                fig.update_traces(xbins=dict(
                    start=bins[0], end=bins[-1],
                    size=(bins[-1]-bins[0])/(len(bins)-1)))
            return self._wrap(fig, title or f'Distribution of {x}')
        except Exception as e:
            return self._err(str(e))




---
# **DataInspector Class**


In [24]:
class DataInspector:
    """End-to-end data sanitization & exploration engine for Google Colab."""

    _NULL_STRINGS = {'?','n/a','na','null','none','',' ','nan',
                     'N/A','NULL','None','NaN','missing','MISSING'}

    def __init__(self):
        self.df = None
        self._plotter = PlottingMethods()

---
# **1 · Data Ingestion & Sanitization**


In [39]:
def upload_data(self):
    """Interactive CSV upload in Colab; sanitizes on load."""
    try:
        from google.colab import files
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]

        self.df = pd.read_csv(
            io.BytesIO(uploaded[fname]),
            na_values=list(self._NULL_STRINGS)
        )

    except ModuleNotFoundError:
        path = input("Enter CSV path: ").strip()

        self.df = pd.read_csv(
            path,
            na_values=list(self._NULL_STRINGS)
        )

    self._sanitize()

    print(
        f"Data loaded: {self.df.shape[0]} rows x "
        f"{self.df.shape[1]} cols"
    )


def _sanitize(self):
    """Replace garbage strings with NaN; auto-convert to numeric."""
    if self.df is None:
        return

    self.df.replace(
        list(self._NULL_STRINGS),
        np.nan,
        inplace=True
    )

    for col in self.df.columns:
        if self.df[col].dtype == object:
            converted = pd.to_numeric(
                self.df[col],
                errors='coerce'
            )

            if converted.notna().sum() > 0:
                self.df[col] = converted

---
# **2 · Structural Analysis & Cleaning**


In [40]:
def get_summary(self):
    """Row/col counts, 20-row preview, numeric vs categorical breakdown."""
    if self._check_empty():
        return

    num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()

    print(f"Shape : {self.df.shape}")
    print(f"Numeric: {len(num_cols)} -> {num_cols}")
    print(f"Categ. : {len(cat_cols)} -> {cat_cols}")

    display(self.df.head(20))
    display(self.df[num_cols].describe())


def column_details(self):
    """dtype, null count, null%, unique count, sample values per column."""
    if self._check_empty():
        return

    info = pd.DataFrame({
        'dtype': self.df.dtypes,
        'nulls': self.df.isnull().sum(),
        'null_%': (self.df.isnull().mean() * 100).round(2),
        'unique': self.df.nunique(),
        'sample': [
            self.df[c].dropna().iloc[:3].tolist()
            for c in self.df.columns
        ]
    })

    display(info)


def get_categorical_summary(self):
    """Value-count table for every categorical column."""
    if self._check_empty():
        return

    for col in self.df.select_dtypes(exclude=np.number).columns:
        print(f"-- {col} --")
        display(self.df[col].value_counts(dropna=False).to_frame())


def show_missing_data(self):
    """Sorted table of columns with missing values."""
    if self._check_empty():
        return

    missing = (
        self.df.isnull().sum()
        .rename('count')
        .to_frame()
        .assign(
            percent=lambda d: (
                d['count'] / len(self.df) * 100
            ).round(2)
        )
        .query('count > 0')
        .sort_values('count', ascending=False)
    )

    if not missing.empty:
        display(missing)
    else:
        print("No missing values.")


def handle_missing_values(self, strategy='mean', constant=0, columns=None):
    """Impute NaNs: strategy = mean | median | mode | constant."""
    if self._check_empty():
        return

    cols = columns if columns else self.df.columns.tolist()
    filled = 0

    for col in cols:
        if self.df[col].isnull().sum() == 0:
            continue

        if strategy == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
            self.df[col].fillna(self.df[col].mean(), inplace=True)

        elif strategy == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
            self.df[col].fillna(self.df[col].median(), inplace=True)

        elif strategy == 'mode':
            self.df[col].fillna(self.df[col].mode().iloc[0], inplace=True)

        elif strategy == 'constant':
            self.df[col].fillna(constant, inplace=True)

        filled += 1

    print(f"Missing values handled ({strategy}) in {filled} column(s).")


def remove_duplicates(self):
    """Remove exact duplicate rows."""
    if self._check_empty():
        return

    before = len(self.df)

    self.df.drop_duplicates(inplace=True)
    self.df.reset_index(drop=True, inplace=True)

    print(
        f"Removed {before - len(self.df)} duplicate(s). "
        f"Remaining: {len(self.df)}"
    )


def handle_outliers(self, columns=None, find_and_delete=False):
    """IQR-based outlier detection; optionally auto-delete rows."""
    if self._check_empty():
        return

    num_cols = self.df.select_dtypes(include=np.number).columns
    cols = [c for c in (columns or num_cols) if c in self.df.columns]

    mask = pd.Series(False, index=self.df.index)

    for col in cols:
        Q1, Q3 = self.df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1

        lo = Q1 - 1.5 * IQR
        hi = Q3 + 1.5 * IQR

        col_mask = (self.df[col] < lo) | (self.df[col] > hi)

        print(
            f"{col}: {col_mask.sum()} outlier(s) "
            f"[{lo:.2f} - {hi:.2f}]"
        )

        mask |= col_mask

    if find_and_delete:
        before = len(self.df)
        self.df = self.df[~mask].reset_index(drop=True)

        print(f"Deleted {before - len(self.df)} outlier row(s).")
    else:
        print(
            f"Total rows flagged: {mask.sum()} "
            f"(pass find_and_delete=True)"
        )


def delete_rows(self, indices=None):
    """Delete rows by comma-separated index string."""
    if self._check_empty():
        return

    if indices is None:
        indices = input(
            "Row indices to delete (comma-separated): "
        )

    idx = [
        int(i.strip())
        for i in indices.split(',')
        if i.strip()
    ]

    self.df.drop(index=idx, inplace=True)
    self.df.reset_index(drop=True, inplace=True)

    print(f"Deleted rows: {idx}")


def delete_columns(self, columns=None):
    """Delete columns by comma-separated name string."""
    if self._check_empty():
        return

    if columns is None:
        columns = input(
            "Column names to delete (comma-separated): "
        )

    cols = [
        c.strip()
        for c in columns.split(',')
        if c.strip()
    ]

    self.df.drop(columns=cols, inplace=True)

    print(f"Deleted columns: {cols}")

---
# **3 · Feature Engineering (Normalization)**


In [41]:
def extract_normalized_numeric_data(self, method='minmax', columns=None):
    """Scale numeric columns: minmax | standard | robust."""
    if self._check_empty():
        return None

    num_cols = list(
        columns or
        self.df.select_dtypes(include=np.number).columns
    )

    scaled = self.df[num_cols].copy()

    for col in num_cols:
        s = scaled[col].dropna()

        if method == 'minmax':
            mn, mx = s.min(), s.max()
            scaled[col] = (
                (scaled[col] - mn) / (mx - mn)
                if mx != mn else 0.0
            )

        elif method == 'standard':
            scaled[col] = (
                (scaled[col] - s.mean()) / s.std()
            )

        elif method == 'robust':
            Q1 = s.quantile(0.25)
            Q3 = s.quantile(0.75)
            IQR = Q3 - Q1

            scaled[col] = (
                (scaled[col] - s.median()) / IQR
                if IQR != 0 else 0.0
            )

    print(
        f"Numeric scaling ({method}) "
        f"on {len(num_cols)} column(s)."
    )

    return scaled


def extract_normalized_categorical_data(
    self,
    method='onehot',
    columns=None,
    ordinal_mapping=None
):
    """Encode categoricals: onehot | ordinal | uniform."""
    if self._check_empty():
        return None

    cat_cols = list(
        columns or
        self.df.select_dtypes(exclude=np.number).columns
    )

    if method == 'onehot':
        encoded = pd.get_dummies(
            self.df[cat_cols],
            drop_first=False
        )

    elif method == 'ordinal':
        encoded = self.df[cat_cols].copy()

        for col in cat_cols:
            if ordinal_mapping and col in ordinal_mapping:
                order = {
                    v: i
                    for i, v in enumerate(
                        ordinal_mapping[col]
                    )
                }
            else:
                cats = encoded[col].dropna().unique()
                order = {
                    v: i
                    for i, v in enumerate(
                        sorted(cats, key=str)
                    )
                }

            encoded[col] = encoded[col].map(order)

    elif method == 'uniform':
        encoded = self.df[cat_cols].copy()

        for col in cat_cols:
            cats = encoded[col].dropna().unique()
            n = len(cats)

            order = {
                v: (
                    i / (n - 1)
                    if n > 1 else 0.0
                )
                for i, v in enumerate(
                    sorted(cats, key=str)
                )
            }

            encoded[col] = encoded[col].map(order)

    else:
        raise ValueError(
            "method must be one of: "
            "onehot, ordinal, uniform"
        )

    print(
        f"Categorical encoding ({method}) "
        f"on {len(cat_cols)} column(s)."
    )

    return encoded


def create_normalized_data_df(
    self,
    num_method='minmax',
    cat_method='onehot'
):
    """Merge scaled numeric + encoded categorical into one DataFrame."""

    scaled = self.extract_normalized_numeric_data(
        method=num_method
    )

    encoded = self.extract_normalized_categorical_data(
        method=cat_method
    )

    parts = [
        p for p in [scaled, encoded]
        if p is not None
    ]

    merged = pd.concat(parts, axis=1)

    print(
        f"Unified normalized DataFrame: "
        f"{merged.shape}"
    )

    return merged

---
# **4 · Advanced Interactive Visualization (Plotly)**

In [42]:
def plot_numerical(self, column_names=None):
    """3-panel subplot per column: Violin/Box | Scatter | Histogram."""
    if self._check_empty():
        return

    num_cols = list(
        column_names or
        self.df.select_dtypes(include=np.number).columns
    )

    for col in num_cols:
        series = self.df[col].dropna()

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                f'{col} Violin/Box',
                f'{col} Scatter',
                f'{col} Histogram'
            ]
        )

        fig.add_trace(
            go.Violin(
                x=series,
                box_visible=True,
                meanline_visible=True,
                orientation='h'
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series,
                mode='markers',
                marker=dict(opacity=0.5)
            ),
            row=1,
            col=2
        )

        fig.add_trace(
            go.Histogram(x=series),
            row=1,
            col=3
        )

        fig.update_layout(
            height=350,
            showlegend=False,
            title_text=f'Distribution – {col}',
            template='plotly_white'
        )

        fig.show()


def plot_relationship(self, col1, col2):
    """
    Auto-selects chart type:
    Num-Num -> Scatter+OLS
    Cat-Num -> Box
    Cat-Cat -> Grouped Bar
    """
    if self._check_empty():
        return

    is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
    is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])

    if is_num1 and is_num2:

        fig = px.scatter(
            self.df,
            x=col1,
            y=col2,
            trendline='ols',
            title=f'{col1} vs {col2}',
            template='plotly_white'
        )

    elif (not is_num1) and is_num2:

        fig = px.box(
            self.df,
            x=col1,
            y=col2,
            points='all',
            template='plotly_white'
        )

    elif is_num1 and (not is_num2):

        fig = px.box(
            self.df,
            x=col2,
            y=col1,
            points='all',
            template='plotly_white'
        )

    else:

        ct = pd.crosstab(
            self.df[col1],
            self.df[col2]
        ).reset_index()

        melted = ct.melt(
            id_vars=col1,
            var_name=col2,
            value_name='count'
        )

        fig = px.bar(
            melted,
            x=col1,
            y='count',
            color=col2,
            barmode='group',
            template='plotly_white'
        )

    fig.show()


def plot_categorical(self, column_names=None):
    """Bar charts with raw counts + percentage labels."""
    if self._check_empty():
        return

    cat_cols = list(
        column_names or
        self.df.select_dtypes(exclude=np.number).columns
    )

    for col in cat_cols:

        vc = self.df[col].value_counts(dropna=False)

        pct = (
            vc / len(self.df) * 100
        ).round(1)

        labels = [
            f"{v}<br>{p}%"
            for v, p in zip(
                vc.values,
                pct.values
            )
        ]

        fig = go.Figure(
            go.Bar(
                x=vc.index.astype(str),
                y=vc.values,
                text=labels,
                textposition='outside',
                marker_color='steelblue'
            )
        )

        fig.update_layout(
            title=f'Frequency – {col}',
            template='plotly_white',
            height=400
        )

        fig.show()

---
# **5 · Deep Statistical Insights**


In [43]:
def plot_numerical_correlation(self):
    """Pearson correlation heatmap for all numeric columns."""
    if self._check_empty():
        return

    corr = self.df.select_dtypes(include=np.number).corr()

    fig = px.imshow(
        corr,
        text_auto='.2f',
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Pearson Correlation',
        template='plotly_white'
    )

    fig.show()


def plot_categorical_correlation(self):
    """Cramer's V heatmap for all categorical column pairs."""
    if self._check_empty():
        return

    cat_cols = self.df.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    def cramers_v(a, b):
        ct = pd.crosstab(
            a.fillna('_'),
            b.fillna('_')
        )

        chi2 = chi2_contingency(
            ct,
            correction=False
        )[0]

        n = ct.sum().sum()
        k = min(ct.shape) - 1

        return np.sqrt(
            chi2 / (n * k)
        ) if k > 0 else 0.0

    mat = pd.DataFrame(
        index=cat_cols,
        columns=cat_cols,
        dtype=float
    )

    for c1 in cat_cols:
        for c2 in cat_cols:
            mat.loc[c1, c2] = cramers_v(
                self.df[c1],
                self.df[c2]
            )

    fig = px.imshow(
        mat.astype(float),
        text_auto='.2f',
        color_continuous_scale='Blues',
        zmin=0,
        zmax=1,
        title="Cramer's V – Categorical",
        template='plotly_white'
    )

    fig.show()


def plot_all_associations_heatmap(self):
    """
    Unified heatmap:
    Pearson r | Cramer's V | Eta coefficient
    """
    if self._check_empty():
        return

    cols = self.df.columns.tolist()
    n = len(cols)

    mat = np.zeros((n, n))

    def is_num(col):
        return pd.api.types.is_numeric_dtype(
            self.df[col]
        )

    def cramers_v(a, b):
        ct = pd.crosstab(
            a.fillna('_'),
            b.fillna('_')
        )

        chi2 = chi2_contingency(
            ct,
            correction=False
        )[0]

        n_ = ct.sum().sum()
        k = min(ct.shape) - 1

        return np.sqrt(
            chi2 / (n_ * k)
        ) if k > 0 else 0.0

    def eta(num_col, cat_col):

        groups = [
            g.dropna().values
            for _, g in self.df.groupby(cat_col)[num_col]
            if len(g) > 0
        ]

        if len(groups) < 2:
            return 0.0

        grand_mean = self.df[num_col].mean()

        ss_between = sum(
            len(g) * (g.mean() - grand_mean) ** 2
            for g in groups
        )

        ss_total = (
            (
                self.df[num_col].dropna()
                - grand_mean
            ) ** 2
        ).sum()

        return (
            np.sqrt(ss_between / ss_total)
            if ss_total > 0
            else 0.0
        )

    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):

            if i == j:
                mat[i, j] = 1.0
                continue

            try:

                if is_num(c1) and is_num(c2):

                    temp = self.df[
                        [c1, c2]
                    ].dropna()

                    r, _ = stats.pearsonr(
                        temp[c1],
                        temp[c2]
                    )

                    mat[i, j] = abs(r)

                elif (not is_num(c1)) and (not is_num(c2)):

                    mat[i, j] = cramers_v(
                        self.df[c1],
                        self.df[c2]
                    )

                else:

                    num_col = (
                        c1 if is_num(c1)
                        else c2
                    )

                    cat_col = (
                        c2 if is_num(c1)
                        else c1
                    )

                    mat[i, j] = eta(
                        num_col,
                        cat_col
                    )

            except Exception:
                mat[i, j] = 0.0

    fig = px.imshow(
        pd.DataFrame(
            mat,
            index=cols,
            columns=cols
        ),
        text_auto='.2f',
        color_continuous_scale='Viridis',
        zmin=0,
        zmax=1,
        title='Unified Association Heatmap',
        template='plotly_white'
    )

    fig.update_layout(height=600)

    fig.show()


def display_image(self, result):
    PlottingMethods.display_image(result)


def _check_empty(self):
    if self.df is None or self.df.empty:
        print(
            "No data loaded. "
            "Call upload_data() first."
        )
        return True

    return False


print(
    "DataInspector and PlottingMethods "
    "defined successfully."
)


DataInspector and PlottingMethods defined successfully.


---
# **Cell 3 — Quick Start Example**


In [44]:
# Initialize the inspector
inspector = DataInspector()

---
# **Cell 4 — Data Import**


In [49]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

inspector = DataInspector()
inspector.df = pd.read_csv(url)

print(
    f"Loaded: "
    f"{inspector.df.shape[0]} rows x "
    f"{inspector.df.shape[1]} columns"
)

print(inspector.df.head())

Loaded: 891 rows x 12 columns
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            37345

---
# **Cell 5 — Initial Data Inspection and Cleaning**


In [53]:
from IPython.display import display
import pandas as pd
import numpy as np


class DataInspector:

    def __init__(self):
        self.df = None

    def _check_empty(self):
        if self.df is None or self.df.empty:
            print("No data loaded. Load data first.")
            return True
        return False

    def get_summary(self):
        """Row/column counts, preview, numeric vs categorical breakdown."""
        if self._check_empty():
            return

        num_cols = self.df.select_dtypes(
            include=np.number
        ).columns.tolist()

        cat_cols = self.df.select_dtypes(
            exclude=np.number
        ).columns.tolist()

        print(f"Shape: {self.df.shape}")
        print(f"Numeric Columns ({len(num_cols)}):")
        print(num_cols)

        print(f"\nCategorical Columns ({len(cat_cols)}):")
        print(cat_cols)

        print("\nFirst 20 Rows:")
        display(self.df.head(20))

        if len(num_cols) > 0:
            print("\nNumeric Summary:")
            display(self.df[num_cols].describe())

    def column_details(self):
        """Column details."""
        if self._check_empty():
            return

        info = pd.DataFrame({
            "dtype": self.df.dtypes,
            "nulls": self.df.isnull().sum(),
            "null_%": (
                self.df.isnull().mean() * 100
            ).round(2),
            "unique": self.df.nunique(),
            "sample": [
                self.df[col]
                .dropna()
                .head(3)
                .tolist()
                for col in self.df.columns
            ]
        })

        display(info)

    def get_categorical_summary(self):
        """Value counts for categorical columns."""
        if self._check_empty():
            return

        cat_cols = self.df.select_dtypes(
            exclude=np.number
        ).columns

        for col in cat_cols:
            print(f"\n--- {col} ---")
            display(
                self.df[col]
                .value_counts(dropna=False)
                .to_frame("count")
            )

---
# **Cell 6 — Feature Engineering & Normalization**


In [56]:
def extract_normalized_numeric_data(
    self,
    method='minmax',
    columns=None
):
    """Scale numeric columns."""
    if self._check_empty():
        return None

    num_cols = list(
        columns or
        self.df.select_dtypes(include=np.number).columns
    )

    scaled = self.df[num_cols].copy()

    for col in num_cols:

        s = scaled[col].dropna()

        if method == 'minmax':

            mn = s.min()
            mx = s.max()

            if mx != mn:
                scaled[col] = (
                    (scaled[col] - mn) /
                    (mx - mn)
                )
            else:
                scaled[col] = 0.0

        elif method == 'standard':

            scaled[col] = (
                scaled[col] - s.mean()
            ) / s.std()

        elif method == 'robust':

            q1 = s.quantile(0.25)
            q3 = s.quantile(0.75)

            iqr = q3 - q1

            if iqr != 0:
                scaled[col] = (
                    scaled[col] - s.median()
                ) / iqr
            else:
                scaled[col] = 0.0

    return scaled


def extract_normalized_categorical_data(
    self,
    method='onehot',
    columns=None,
    ordinal_mapping=None
):
    """Encode categorical columns."""
    if self._check_empty():
        return None

    cat_cols = list(
        columns or
        self.df.select_dtypes(
            exclude=np.number
        ).columns
    )

    if method == 'onehot':

        encoded = pd.get_dummies(
            self.df[cat_cols],
            drop_first=False
        )

    elif method == 'ordinal':

        encoded = self.df[cat_cols].copy()

        for col in cat_cols:

            if (
                ordinal_mapping and
                col in ordinal_mapping
            ):
                mapping = {
                    v: i
                    for i, v in enumerate(
                        ordinal_mapping[col]
                    )
                }
            else:
                cats = sorted(
                    encoded[col]
                    .dropna()
                    .unique(),
                    key=str
                )

                mapping = {
                    v: i
                    for i, v in enumerate(cats)
                }

            encoded[col] = (
                encoded[col].map(mapping)
            )

    else:
        raise ValueError(
            "method must be "
            "'onehot' or 'ordinal'"
        )

    return encoded


def create_normalized_data_df(
    self,
    num_method='minmax',
    cat_method='onehot'
):
    """Combine numeric and categorical data."""

    numeric = (
        self.extract_normalized_numeric_data(
            method=num_method
        )
    )

    categorical = (
        self.extract_normalized_categorical_data(
            method=cat_method
        )
    )

    return pd.concat(
        [numeric, categorical],
        axis=1
    )

---
# **Cell 7 — Data Association Plots**


In [59]:
def plot_numerical_correlation(self):
    """Pearson correlation heatmap for numeric columns."""
    if self._check_empty():
        return

    corr = self.df.select_dtypes(include=np.number).corr()

    fig = px.imshow(
        corr,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title="Pearson Correlation Heatmap",
        template="plotly_white"
    )

    fig.show()


def plot_categorical_correlation(self):
    """Cramer's V heatmap for categorical columns."""
    if self._check_empty():
        return

    cat_cols = self.df.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    def cramers_v(a, b):
        ct = pd.crosstab(
            a.fillna("_"),
            b.fillna("_")
        )

        chi2 = chi2_contingency(
            ct,
            correction=False
        )[0]

        n = ct.sum().sum()
        k = min(ct.shape) - 1

        return np.sqrt(chi2 / (n * k)) if k > 0 else 0.0

    mat = pd.DataFrame(
        index=cat_cols,
        columns=cat_cols,
        dtype=float
    )

    for c1 in cat_cols:
        for c2 in cat_cols:
            mat.loc[c1, c2] = cramers_v(
                self.df[c1],
                self.df[c2]
            )

    fig = px.imshow(
        mat,
        text_auto=".2f",
        color_continuous_scale="Blues",
        zmin=0,
        zmax=1,
        title="Cramer's V Heatmap",
        template="plotly_white"
    )

    fig.show()

---
# **Cell 8 — Custom Plotting (PlottingMethods)**


In [60]:
# ==========================
# Initialize Plotting Class
# ==========================

PLT = PlottingMethods()

# ==========================
# Get Available Methods Info
# ==========================

response = PLT.get_methods_info()

import pandas as pd
pd.DataFrame(response.get('response', {}))

# ==========================
# Bar Chart: Pclass vs Survived
# ==========================

result = PLT.plot_bar_chart(
    x='Pclass',
    y='Survived',
    color='Sex',
    barmode='group',
    data=inspector.df
)

PLT.display_image(result=result)

# ==========================
# Pie Chart: Sex Distribution
# ==========================

sex_counts = inspector.df['Sex'].value_counts().reset_index()
sex_counts.columns = ['Sex', 'Count']

result = PLT.plot_pie_chart(
    names='Sex',
    values='Count',
    hole=0.4,
    title='Passenger Sex Distribution',
    data=sex_counts
)

PLT.display_image(result=result)

# ==========================
# Histogram: Age Distribution
# ==========================

result = PLT.plot_histogram(
    x='Age',
    bins=[0, 18, 35, 60, 100],
    title='Age Demographics',
    data=inspector.df
)

PLT.display_image(result=result)